In [ ]:
FIGURE = 'Fig.3'
from pathlib import Path
from contextlib import contextmanager
import os
import sys
import gc
import mmap
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
import netCDF4 as nc
_NETCDF_DATASET = nc.Dataset
HERE = Path.cwd().resolve()
if not (HERE / 'Inputs').is_dir():
    for candidate in [HERE / 'Figure' / FIGURE, HERE / FIGURE, HERE / 'Data Processing' if FIGURE in ['SSMIrz','Rmax_Ts'] else HERE]:
        if (candidate / 'Inputs').is_dir():
            HERE = candidate
            break
INPUT_DIR = HERE / 'Inputs'
OUTPUT_DIR = HERE / 'Outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
ROOT = HERE
DATA = INPUT_DIR
OUTPUT = OUTPUT_DIR
__file__ = str(HERE / (FIGURE + '.ipynb'))
sys.argv = ['notebook']

def open_nc(path, mode='r', **kwargs):
    if 'memory' in kwargs:
        return _NETCDF_DATASET(path, mode, **kwargs)
    path = Path(path).resolve()
    previous = Path.cwd()
    try:
        os.chdir(path.parent)
        return _NETCDF_DATASET(path.name, mode, **kwargs)
    finally:
        os.chdir(previous)

def asset_path(key):
    if key in ['asset_e2603c619a67', 'asset_5e3c32038bfb']:
        return INPUT_DIR / 'study_mask.tif'
    if key == 'asset_beb53c9b3525':
        return INPUT_DIR / 'SPEI_k01-k24.nc'
    if key == 'asset_174bc324d489':
        return INPUT_DIR / 'SSMIrz_k01-k24.nc'
    return OUTPUT_DIR

def output_path(name=''):
    path = OUTPUT_DIR / name
    path.mkdir(parents=True, exist_ok=True)
    return path

def module_path(name=''):
    return str(HERE)

def font_path(filename):
    from matplotlib.font_manager import FontProperties, findfont
    family = 'Times New Roman' if filename.lower().startswith('times') else 'Courier New' if filename.lower().startswith('cour') else 'Arial'
    return findfont(FontProperties(family=family))

def array(value):
    return np.asarray(np.ma.filled(value.astype(float), np.nan))

@contextmanager
def mapped(path):
    with Path(path).open('rb') as stream:
        mm = mmap.mmap(stream.fileno(), 0, access=mmap.ACCESS_READ)
        dataset = open_nc('input.nc', memory=mm)
        try:
            yield dataset
        finally:
            dataset.close()
            gc.collect()
            mm.close()


In [ ]:
def draw_figure3(metric):
    METRIC = metric
    assert METRIC in {'Rmax', 'Ts'}
    OUT = output_path('Fig.3')
    file_spei = str(INPUT_DIR / f'SPEI_{METRIC}_500replicates60pct.xlsx')
    file_smrz = str(INPUT_DIR / f'SSMIrz_{METRIC}_500replicates60pct.xlsx')
    out_png = str(OUT / f'Fig3_{METRIC}_600dpi.png')
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    import matplotlib.gridspec as gridspec
    import re
    plt.rcParams.update({'font.family': 'Arial', 'font.size': 11, 'axes.linewidth': 0.6, 'figure.facecolor': 'white', 'axes.facecolor': 'white'})
    type_color_map = {'VPD': '#BDBDBD', 'SRAD': '#E6E6E6', 'LAI': '#BDBDBD', 'CO2': '#E6E6E6', 'PRE': '#BDBDBD', 'TEM': '#E6E6E6'}
    RANK_FILL_RATIO = {1: 0.82, 2: 0.63, 3: 0.44}
    COLOR_SPEI = '#23025c' if METRIC == 'Rmax' else '#2b0076'
    COLOR_SMRZ = '#00b2cb' if METRIC == 'Rmax' else '#A6CFCD'
    MARKER_SPEI = 'o'
    MARKER_SMRZ = 'D'
    BUBBLE_EDGE_COLOR = 'white'
    BUBBLE_EDGE_LW = 0.5
    OFFSET_X = 0.0
    OFFSET_Y = 0.0
    cell_edge_color = '#E6E6E6'
    cell_edge_lw = 0.6
    group_line_color = '#BDBDBD'
    group_line_lw = 1.2
    climates = ['Temperate']
    seasons = ['MAM']
    gap_rows = 1
    metric_order = ('mean', 'trend', 'cv')
    bar_height_ratio = 0.3
    heat_height_ratio = 6.0
    legend_ratio = 0.28
    bar_group_width = 0.86
    bar_gap = 0
    bar_single_width = (bar_group_width - bar_gap) / 2
    base_color = '#E9EEF2'
    show_bar_grid = False
    shift = -0.14
    gap = -0.165
    DIAMOND_MS_SCALE = 0.72

    def format_label(label: str) -> str:
        s = str(label)
        s = s.replace('_Y_', ' Y_')
        s = s.replace('CO2', 'CO$_2$')
        s = s.replace('SMrz', 'SM$_{rz}$')
        return s

    def parse_var_name(v: str):
        s = str(v).strip()
        m = re.match('^(?P<driver>.+?)(?P<y>_Y)?_(?P<metric>Mean|Trend|CV)$', s, flags=re.IGNORECASE)
        if not m:
            return (s, False, None)
        return (m.group('driver'), m.group('y') is not None, m.group('metric').lower())

    def get_var_prefix(var_name: str):
        driver, _, _ = parse_var_name(str(var_name))
        for p in type_color_map:
            if driver.startswith(p):
                return p
        return 'OTHER'

    def read_rank_excel(path):
        df = pd.read_excel(path, header=[0, 1])
        variables = df.iloc[:, 0].astype(str).values
        rows = []
        for i, var in enumerate(variables):
            for j in range(1, len(df.columns)):
                v = df.iloc[i, j]
                if pd.notna(v) and str(v).strip() != '':
                    try:
                        rk = int(float(v))
                        if rk in (1, 2, 3):
                            rows.append({'variable': str(var), 'climate': str(df.columns[j][0]).strip(), 'season': str(df.columns[j][1]).strip(), 'rank': rk})
                    except:
                        pass
        return (pd.DataFrame(rows), variables)

    def build_y_layout(seasons, climates, gap_rows=1):
        y_pos, y_to_climate, season_centers = ({}, {}, {})
        y = 0
        for si, s in enumerate(seasons):
            start_y = y
            for c in climates:
                y_pos[s, c] = y
                y_to_climate[y] = c
                y += 1
            season_centers[s] = (start_y + y - 1) / 2
            if si < len(seasons) - 1:
                for _ in range(gap_rows):
                    y_to_climate[y] = None
                    y += 1
        return (y_pos, y_to_climate, season_centers, y)

    def sort_and_group_variables(vars_array, metric_order=('mean', 'trend', 'cv')):
        vars_list = [str(v) for v in vars_array]
        driver_order, seen = ([], set())
        for v in vars_list:
            d, _, _ = parse_var_name(v)
            if d not in seen:
                seen.add(d)
                driver_order.append(d)
        metric_rank = {m: i for i, m in enumerate(metric_order)}

        def key(v):
            d, iy, m = parse_var_name(v)
            return (driver_order.index(d) if d in driver_order else 1000000000.0, 1 if iy else 0, metric_rank.get(m, 1000000.0), v)
        vs = sorted(vars_list, key=key)
        groups = []
        for d in driver_order:
            idxs = [i for i, v in enumerate(vs) if parse_var_name(v)[0] == d]
            if idxs:
                groups.append((d, min(idxs), max(idxs)))
        return (vs, groups)

    def count_ranks(variables_sorted, climates, seasons, data_map, ranks=(1, 2, 3)):
        counts = {v: {rk: 0 for rk in ranks} for v in variables_sorted}
        for v in variables_sorted:
            for s in seasons:
                for c in climates:
                    r = data_map.get((v, c, s))
                    if r in ranks:
                        counts[v][r] += 1
        return counts

    def compute_s_values(ax, fig, rank_fill_ratio):
        fig.canvas.draw()
        ax_width_inch = ax.get_position().width * fig.get_figwidth()
        ax_width_pts = ax_width_inch * 72.0
        xlim = ax.get_xlim()
        x_data_range = xlim[1] - xlim[0]
        cell_pts = ax_width_pts / x_data_range
        s_dict = {}
        for rk, ratio in rank_fill_ratio.items():
            d_pts = ratio * cell_pts
            s_dict[rk] = np.pi * (d_pts / 2) ** 2
        return (s_dict, cell_pts)
    df_spei, variables_spei = read_rank_excel(file_spei)
    df_smrz, variables_smrz = read_rank_excel(file_smrz)
    if list(map(str, variables_spei)) != list(map(str, variables_smrz)):
        raise ValueError('Variable sets differ between the two tables!')
    variables_sorted, var_groups = sort_and_group_variables(variables_spei, metric_order=metric_order)
    nx = len(variables_sorted)
    y_pos, y_to_climate, season_centers, ny = build_y_layout(seasons, climates, gap_rows=gap_rows)
    spei_map = {(r.variable, r.climate, r.season): int(r.rank) for r in df_spei.itertuples()}
    smrz_map = {(r.variable, r.climate, r.season): int(r.rank) for r in df_smrz.itertuples()}
    counts_spei = count_ranks(variables_sorted, climates, seasons, spei_map)
    counts_smrz = count_ranks(variables_sorted, climates, seasons, smrz_map)
    fig_w = max(9.0, nx * 0.28)
    fig_h = max(5.8, ny * 0.42 + 1.5)
    fig = plt.figure(figsize=(fig_w, fig_h), dpi=600)
    gs = gridspec.GridSpec(3, 1, height_ratios=[legend_ratio, bar_height_ratio, heat_height_ratio], hspace=0.0)
    ax_legend = fig.add_subplot(gs[0])
    ax_bar = fig.add_subplot(gs[1])
    ax = fig.add_subplot(gs[2], sharex=ax_bar)
    ax_legend.axis('off')
    x_centers = np.arange(nx) + 0.5
    x_left = x_centers - (bar_gap / 2 + bar_single_width / 2)
    x_right = x_centers + (bar_gap / 2 + bar_single_width / 2)
    totals_s = np.array([sum(counts_spei[v].values()) for v in variables_sorted])
    totals_m = np.array([sum(counts_smrz[v].values()) for v in variables_sorted])
    ymax_bar = int(max(totals_s.max(initial=0), totals_m.max(initial=0), 1))
    ax_bar.bar(x_left, np.full(nx, ymax_bar), width=bar_single_width, color=base_color, edgecolor='none', zorder=0)
    ax_bar.bar(x_right, np.full(nx, ymax_bar), width=bar_single_width, color=base_color, edgecolor='none', zorder=0)
    rank_alphas = {1: 1.0, 2: 0.7, 3: 0.4}
    bot_l = np.zeros(nx)
    bot_r = np.zeros(nx)
    for rk in [1, 2, 3]:
        vl = np.array([counts_spei[v][rk] for v in variables_sorted], dtype=float)
        vr = np.array([counts_smrz[v][rk] for v in variables_sorted], dtype=float)
        ax_bar.bar(x_left, vl, bottom=bot_l, width=bar_single_width, color=COLOR_SPEI, alpha=rank_alphas[rk], edgecolor='none', zorder=2)
        ax_bar.bar(x_right, vr, bottom=bot_r, width=bar_single_width, color=COLOR_SMRZ, alpha=rank_alphas[rk], edgecolor='none', zorder=2)
        bot_l += vl
        bot_r += vr
    ax_bar.set_ylim(0, ymax_bar)
    ax_bar.set_ylabel('Count', fontsize=11)
    ax_bar.set_yticks([0, ymax_bar])
    ax_bar.set_yticklabels(['0', str(ymax_bar)], fontsize=11)
    ax_bar.tick_params(axis='x', which='both', bottom=False, labelbottom=False, length=0)
    ax_bar.tick_params(axis='y', length=0)
    for sp in ax_bar.spines.values():
        sp.set_visible(False)
    if show_bar_grid:
        ax_bar.grid(axis='y', color='#BDBDBD', lw=0.6, alpha=0.6)
    for _, sx, ex in var_groups:
        xl = ex + 1
        if xl < nx:
            ax_bar.plot([xl, xl], [0, ymax_bar], color=group_line_color, lw=group_line_lw, zorder=5)
    for s in seasons:
        for c in climates:
            y = y_pos[s, c]
            for x in range(nx):
                ax.add_patch(Rectangle((x, y), 1, 1, facecolor='white', edgecolor=cell_edge_color, linewidth=cell_edge_lw, zorder=0))
    for y in range(ny):
        if y_to_climate.get(y) is None:
            for x in range(nx):
                ax.add_patch(Rectangle((x, y), 1, 1, facecolor='white', edgecolor='white', linewidth=0, zorder=0))
    right_pad = 1.2
    ax.set_xlim(0, nx + right_pad)
    ax.set_ylim(-0.7, ny)
    ax.set_aspect('equal')
    ax_bar.set_xlim(0, nx + right_pad)
    for _, sx, ex in var_groups:
        xl = ex + 1
        if xl < nx:
            ax.plot([xl, xl], [0, ny], color=group_line_color, lw=group_line_lw, zorder=6)
    for gname, sx, ex in var_groups:
        prefix = get_var_prefix(gname)
        if prefix in type_color_map:
            ax.add_patch(Rectangle((sx, -0.55), ex - sx + 1, 0.18, facecolor=type_color_map[prefix], edgecolor='none', alpha=0.35, zorder=1))
    ax.set_xticks(np.arange(nx) + 0.5)
    ax.set_xticklabels([format_label(v) for v in variables_sorted], fontsize=10, rotation=90, va='top')
    yticks, ylabels = ([], [])
    for y in range(ny):
        c = y_to_climate.get(y)
        if c:
            yticks.append(y + 0.5)
            ylabels.append(c)
    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels, fontsize=11)
    for s in seasons:
        ax.text(nx + 0.55, season_centers[s] + 0.5, s, ha='center', va='center', rotation=270, fontsize=11, fontweight='bold', color='#2b2b2b')
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.tick_params(length=0)
    plt.tight_layout(pad=0.4, h_pad=0.0, w_pad=0.0)
    fig.subplots_adjust(hspace=0.02)
    pos_bar = ax_bar.get_position()
    pos_ax = ax.get_position()
    ax_bar.set_position([pos_bar.x0, pos_bar.y0 - shift + gap, pos_bar.width, pos_bar.height])
    ax.set_position([pos_ax.x0, pos_ax.y0 - shift, pos_ax.width, pos_ax.height])
    rank_s, cell_pts = compute_s_values(ax, fig, RANK_FILL_RATIO)
    rank_s_diamond = {rk: v * DIAMOND_MS_SCALE ** 2 for rk, v in rank_s.items()}

    def _draw_bubbles_per_cell(ax, spei_map, smrz_map, rank_s, rank_s_diamond, variables_sorted, seasons, climates, y_pos):
        for xi, var in enumerate(variables_sorted):
            for s in seasons:
                for c in climates:
                    cx = xi + 0.5 + OFFSET_X
                    cy = y_pos[s, c] + 0.5 + OFFSET_Y
                    items = []
                    r_spei = spei_map.get((var, c, s), None)
                    if r_spei in (1, 2, 3):
                        items.append({'source': 'spei', 'rank': r_spei, 'size': rank_s[r_spei], 'marker': MARKER_SPEI, 'color': COLOR_SPEI})
                    r_smrz = smrz_map.get((var, c, s), None)
                    if r_smrz in (1, 2, 3):
                        items.append({'source': 'smrz', 'rank': r_smrz, 'size': rank_s_diamond[r_smrz], 'marker': MARKER_SMRZ, 'color': COLOR_SMRZ})
                    items.sort(key=lambda d: (d['rank'], 0 if d['source'] == 'spei' else 1))
                    base_z = 3
                    for i, item in enumerate(items):
                        ax.scatter([cx], [cy], s=item['size'], c=item['color'], marker=item['marker'], edgecolors=BUBBLE_EDGE_COLOR, linewidths=BUBBLE_EDGE_LW, zorder=base_z + i, alpha=0.92)
    _draw_bubbles_per_cell(ax, spei_map, smrz_map, rank_s, rank_s_diamond, variables_sorted, seasons, climates, y_pos)

    def draw_constrained_legend(ax_leg, ax_ref, fig, rank_s, rank_s_diamond, nx):
        fig.canvas.draw()
        p = ax_ref.get_position()
        b = ax_bar.get_position()
        gap_actual = b.y0 - p.y1
        height = 0.04
        ax_leg.set_position([p.x0, b.y1 + gap_actual, p.width * nx / (nx + right_pad), height])
        ax_leg.set_xlim(0, 1)
        ax_leg.set_ylim(0, 1)
        ax_leg.axis('off')
        items = []
        items.append(('text', '$R_{\\mathrm{max}}$:' if METRIC == 'Rmax' else 'Ts:', None))
        items.append(('text', 'SPEI', None))
        for r in (1, 2, 3):
            items.append(('rank', f'Rank {r}', ('o', COLOR_SPEI, rank_s[r])))
        items.append(('text', 'SSMI$_{\\mathrm{rz}}$', None))
        for r in (1, 2, 3):
            items.append(('rank', f'Rank {r}', ('D', COLOR_SMRZ, rank_s_diamond[r])))
        renderer = fig.canvas.get_renderer()
        widths = []
        for kind, label, style in items:
            t = ax_leg.text(0, 0.5, label, fontsize=11 if kind == 'text' else 10, va='center', fontweight='bold' if kind == 'text' else 'normal')
            fig.canvas.draw()
            w = t.get_window_extent(renderer).width
            t.remove()
            markerwidth = 1.2 * ax_leg.get_window_extent(renderer).width / nx if style else 0
            widths.append((w + markerwidth, markerwidth))
        W = ax_leg.get_window_extent(renderer).width
        spacing = (W - sum((w for w, m in widths))) / (len(items) - 1)
        assert spacing > 0
        x = 0
        for (kind, label, style), (w, m) in zip(items, widths):
            if style:
                marker, color, size = style
                ax_leg.scatter([(x + m / 2) / W], [0.5], s=size, c=color, marker=marker, edgecolors='white', linewidths=0.5)
            ax_leg.text((x + m) / W, 0.5, label, fontsize=11 if kind == 'text' else 10, va='center', ha='left', fontweight='bold' if kind == 'text' else 'normal')
            x += w + spacing
        fig.canvas.draw()

        def visible_bottom():
            bottoms = [t.get_window_extent(fig.canvas.get_renderer()).y0 for t in ax_leg.texts]
            for collection in ax_leg.collections:
                centers = collection.get_offset_transform().transform(collection.get_offsets())
                path = collection.get_paths()[0].get_extents()
                scale = float(collection.get_sizes()[0]) ** 0.5 * fig.dpi / 72
                bottoms.extend(centers[:, 1] + path.y0 * scale - collection.get_linewidths()[0] * fig.dpi / 144)
            return min(bottoms)
        legend_gap = gap_actual * 0.75
        target_bottom = (b.y1 + legend_gap) * fig.bbox.height
        pos = ax_leg.get_position()
        ax_leg.set_position([pos.x0, pos.y0 + (target_bottom - visible_bottom()) / fig.bbox.height, pos.width, pos.height])
        fig.canvas.draw()
        import json
        actual_gap = visible_bottom() / fig.bbox.height - b.y1
        assert abs(actual_gap - legend_gap) < 1e-10
        (OUT / (METRIC + '_layout_validation.json')).write_text(json.dumps({'main_to_bar_gap': gap_actual, 'bar_to_legend_visible_gap': actual_gap, 'legend_width': ax_leg.get_position().width, 'graphic_width': p.width * nx / (nx + right_pad), 'dpi': 600}, indent=2), encoding='utf-8')
    draw_constrained_legend(ax_legend, ax, fig, rank_s, rank_s_diamond, nx)
    plt.savefig(out_png, dpi=600, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'✅ saved: {out_png}')
    print('Rank s (points²):', {rk: f'{v:.1f}' for rk, v in rank_s.items()})
    print('cell_pts:', f'{cell_pts:.2f} pts')
for metric in ['Rmax', 'Ts']:
    draw_figure3(metric)
